# Stage 2 — Cold-start SFT

Take a base model and teach it the **format** with a tiny supervised dataset. We're not trying to make it smart yet — we're teaching it to write `T … A …` (think then answer) every time.

Notebooks 03 → 04 → 05 share **one toy task**: 1-digit addition with a scratchpad. Watch the pass-rate climb stage by stage.

**Format we're teaching:** `3+4=T3+4 A7.` — `T` opens the scratchpad, `A` opens the answer, `.` ends.

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F, random
torch.manual_seed(0); random.seed(0)

# Vocab: digits, operators, format markers, space, EOS '.'
VOCAB = list('0123456789+=TA .')
stoi = {c: i for i, c in enumerate(VOCAB)}
itos = {i: c for c, i in stoi.items()}
V = len(VOCAB)
PAD = stoi[' ']
EOS = stoi['.']

def encode(s): return [stoi[c] for c in s]
def decode(ids): return ''.join(itos[i] for i in ids)
print('vocab:', VOCAB, '| size:', V)

vocab: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '+', '=', 'T', 'A', ' ', '.'] | size: 16


In [2]:
# Build the SFT dataset: 20 hand-formatted examples covering 1-digit addition
def make_example(a, b):
    return f'{a}+{b}=T{a}+{b} A{a+b}.'

examples = [make_example(a, b) for a in range(10) for b in range(10)]
random.shuffle(examples)
sft_data = examples[:20]  # cold-start: just 20!
for ex in sft_data[:5]: print(ex)

2+3=T2+3 A5.
0+8=T0+8 A8.
1+1=T1+1 A2.
0+7=T0+7 A7.
4+8=T4+8 A12.


In [3]:
BLOCK = 24
class TinyLM(nn.Module):
    def __init__(self, V, d=64, h=4, L=2, block=BLOCK):
        super().__init__()
        self.tok = nn.Embedding(V, d); self.pos = nn.Embedding(block, d)
        self.blocks = nn.ModuleList([nn.TransformerEncoderLayer(d, h, dim_feedforward=4*d, batch_first=True, activation='gelu') for _ in range(L)])
        self.head = nn.Linear(d, V); self.block = block
    def forward(self, x):
        T = x.shape[1]
        h = self.tok(x) + self.pos(torch.arange(T, device=x.device))
        mask = nn.Transformer.generate_square_subsequent_mask(T).to(x.device)
        for blk in self.blocks:
            h = blk(h, src_mask=mask)
        return self.head(h)

model = TinyLM(V)
print('params:', sum(p.numel() for p in model.parameters())/1e3, 'K')

params: 103.568 K


In [4]:
def pad_batch(strings):
    ids = [encode(s) for s in strings]
    L = max(len(x) for x in ids)
    return torch.tensor([x + [PAD]*(L-len(x)) for x in ids])

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
for step in range(400):
    batch = random.choices(sft_data, k=16)
    x = pad_batch(batch)
    logits = model(x[:, :-1])
    loss = F.cross_entropy(logits.reshape(-1, V), x[:, 1:].reshape(-1), ignore_index=PAD)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 80 == 0: print(f'step {step:3d}  loss {loss.item():.3f}')

step   0  loss 2.872


step  80  loss 0.198


step 160  loss 0.105


step 240  loss 0.139


step 320  loss 0.103


In [5]:
# Save the model so notebook 04 can load it as the cold-start checkpoint
torch.save(model.state_dict(), 'cold_start.pt')
print('saved cold_start.pt')

saved cold_start.pt


In [6]:
@torch.no_grad()
def generate(prompt, max_new=16):
    x = torch.tensor([encode(prompt)])
    for _ in range(max_new):
        logits = model(x[:, -BLOCK:])[:, -1, :]
        nxt = torch.argmax(logits, -1, keepdim=True)
        x = torch.cat([x, nxt], 1)
        if nxt.item() == EOS: break
    return decode(x[0].tolist())

import re
def extract_answer(s):
    m = re.search(r'A(\d+)\.', s)
    return int(m.group(1)) if m else None

def pass_rate(n=50):
    correct = 0
    for _ in range(n):
        a, b = random.randint(0,9), random.randint(0,9)
        out = generate(f'{a}+{b}=')
        if extract_answer(out) == a + b: correct += 1
    return correct / n

print('sample:', generate('3+4='))
print('sample:', generate('7+8='))
print(f'pass-rate after cold-start SFT: {pass_rate():.0%}')

sample: 3+4=T4+35.
sample: 7+8=T7+88A15.
pass-rate after cold-start SFT: 4%


## Takeaway

20 examples is enough to lock in the **format** — every output now looks like `T … A …`.

But the actual answer is often wrong: the model learned the *shape*, not the *skill*. That's exactly what RL is going to fix in notebook 04.

Track this pass-rate. We'll see it climb.